# September 15 export comparison and scoped Q1 walkthrough

The comparison was reported before choosing the Q1 extension. June artifacts remain untouched. Source scripts beside this notebook contain the exact operations; each script writes only to this dated directory. See `comparison.md` for the completion rules and duplicate limitations.

In [1]:
from pathlib import Path
import json, runpy
snapshot_dir = Path.cwd()
if not (snapshot_dir / 'compare_exports.py').exists():
    snapshot_dir = snapshot_dir / 'analysis/snapshots/2026-09-15'
assert (snapshot_dir / 'compare_exports.py').exists()
_ = runpy.run_path(str(snapshot_dir / 'compare_exports.py'), run_name='__main__')

{
  "delta": {
    "added_rows": 7763,
    "removed_rows": 0,
    "all_old_rows_preserved_with_multiplicity": true,
    "new_distinct_exported_signatures": 7576,
    "increased_multiplicity_of_old_signatures": 0,
    "duplicate_excess_among_additions": 187,
    "added_events_by_type": {
      "app_start": 9,
      "arrow_pan": 211,
      "back_step": 8,
      "cell_click": 1443,
      "idle_end": 22,
      "idle_start": 30,
      "label_select": 163,
      "reset": 1,
      "slide_load": 170,
      "slide_next": 147,
      "viewport_poll": 2556,
      "zoom_step": 3003
    },
    "added_duplicate_excess_by_event": {
      "app_start": 0,
      "arrow_pan": 0,
      "back_step": 0,
      "cell_click": 1,
      "idle_end": 0,
      "idle_start": 1,
      "label_select": 0,
      "reset": 0,
      "slide_load": 0,
      "slide_next": 0,
      "viewport_poll": 3,
      "zoom_step": 182
    },
    "first_added_event_utc": "2026-07-02T13:44:41+00:00",
    "last_added_event_utc": "2026-09-07T

## Comparison decision

Exact complete-reader histories are unchanged. A full Q1-Q6 refresh is unnecessary. Q1 alone is checked against the preserved complete-reader results and extended descriptively to Rana. No partial reader is pooled into the complete-reader cohort.

In [2]:
_ = runpy.run_path(str(snapshot_dir / 'q1_walkthrough.py'), run_name='__main__')

{
  "snapshot": "2026-09-15",
  "question": "How often does a reader's slide-level diagnosis match the reference label?",
  "label_rule": "Existing metrics.slide_level_diagnoses(use_fallback=True)",
  "complete_readers_250_slides": {
    "muhammad.aslam": {
      "n": 250,
      "three_class_correct": 163,
      "three_class_accuracy": 0.652,
      "binary_correct": 217,
      "binary": {
        "n": 250,
        "tp": 163,
        "fp": 17,
        "fn": 16,
        "tn": 54,
        "accuracy": 0.868,
        "precision": 0.9055555555555556,
        "recall": 0.9106145251396648,
        "specificity": 0.7605633802816901,
        "fp_rate": 0.23943661971830985,
        "fn_rate": 0.0893854748603352
      },
      "grade_disagreements_hidden_by_binary_collapse": 54,
      "confusion_order": [
        "non-neoplastic",
        "low-grade",
        "high-grade"
      ],
      "confusion_rows_reference_columns_reader": [
        [
          54,
          17,
          0
        ],
      

## Q1: purpose, method, result and limitations

**Purpose:** describe agreement with the dataset reference diagnosis.

**Method:** one final extracted diagnosis per reader and slide, using the existing fallback helper. Count exact three-class matches. Separately collapse low-grade and high-grade into the same positive category and count binary matches. Neither events nor repeated attempts enlarge the slide denominator.

**Result:** the following table keeps complete and partial populations explicit. The source JSON also provides confusion counts, the strict-label sensitivity and a same-180-slide comparison.

**Limitations:** reference agreement is not independent proof of clinical truth; binary collapse hides grade disagreements; Rana has 70 unlabelled slides; only two readers cover the complete cohort; patient grouping is unresolved. No causal or population-wide reader ranking is inferred.

In [3]:
import pandas as pd
q1 = json.loads((snapshot_dir / 'q1-results.json').read_text(encoding='utf-8'))
rows = []
for reader, s in q1['complete_readers_250_slides'].items():
    rows.append({'reader': reader, 'cohort': 'complete', 'slides': s['n'], 'exact_matches': s['three_class_correct'], 'binary_matches': s['binary_correct']})
s = q1['partial_reader_rana_180_slides']
rows.append({'reader': 'iftikhar.rana', 'cohort': 'partial', 'slides': s['n'], 'exact_matches': s['three_class_correct'], 'binary_matches': s['binary_correct']})
pd.DataFrame(rows)

Out[3]: 
           reader    cohort  slides  exact_matches  binary_matches
0  muhammad.aslam  complete     250            163             217
1   ashish.bansal  complete     250            176             221
2   iftikhar.rana   partial     180            126             157


## Sources

- The unchanged June 29 export and the separately copied September 15 export, with hashes in `comparison.json`.
- `analysis/src/load.py`, `labels.py`, `metrics.py` and the canonical `analysis/notebooks/pathologist_insights.py`.
- `analysis/reports/2026-09-04-reproduction-and-comparison-audit.md`.
- The preserved `analysis/reports/results.json`, used only to validate complete-reader Q1.

Detailed slide/attempt evidence remains in the ignored, date-labelled CSVs beside this notebook.